<a href="https://colab.research.google.com/github/Krod916/bch-intelligence-engine/blob/main/BCH_Full_Intelligence_Command_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BCH Full Intelligence Control System
## Pipeline Reader → Position Cleaner → Greeks Engine → Exit Engine → Heat Limit Engine → BCH Command Dashboard

This notebook turns broker/order-history data into a BCH book-level command dashboard. It is designed to stop closed spreads from being misclassified as open, calculate portfolio Greeks, rank exits, and produce a daily command queue.

In [1]:
# ============================================================
# BCH FULL INTELLIGENCE CONTROL SYSTEM
# Pipeline Reader → Position Cleaner → Greeks Engine → Exit Engine
# → Heat Limit Engine → BCH Command Dashboard
# ============================================================

import math
import re
import numpy as np
import pandas as pd
from datetime import datetime, date
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

TODAY = pd.Timestamp.today().normalize()
RISK_FREE_RATE = 0.045   # editable
DEFAULT_IV = 0.55        # editable fallback when live IV is unavailable
CONTRACT_MULTIPLIER = 100

print('BCH Intelligence Engine loaded:', TODAY.date())

BCH Intelligence Engine loaded: 2026-05-30


## 1. Pipeline Reader
Upload a Fidelity CSV/export when available. If no file is uploaded, the notebook uses a BCH sample book so the engine runs immediately.

In [2]:
# ============================================================
# 1. PIPELINE READER
# ============================================================

def try_colab_upload():
    """Optional upload helper. In Colab, uncomment and run to upload Fidelity CSV files."""
    try:
        from google.colab import files
        uploaded = files.upload()
        return list(uploaded.keys())
    except Exception:
        return []


def read_pipeline_file(path=None):
    """Reads CSV/XLSX/TXT input. Returns raw dataframe or raw text dataframe."""
    if path is None:
        return None
    p = str(path)
    if p.lower().endswith('.csv'):
        return pd.read_csv(p)
    if p.lower().endswith(('.xlsx', '.xls')):
        return pd.read_excel(p)
    if p.lower().endswith(('.txt', '.log')):
        with open(p, 'r', encoding='utf-8', errors='ignore') as f:
            txt = f.read()
        return pd.DataFrame({'raw_text':[txt]})
    raise ValueError('Unsupported file type. Use CSV, XLSX, or TXT.')


def sample_bch_book():
    """Replace this with broker export data once connected."""
    return pd.DataFrame([
        # CLOSED NVDA spread from the May 29, 2026 execution example
        dict(trade_id='NVDA_20260821_180_205_BCS', symbol='NVDA', expiration='2026-08-21', structure='BULL_CALL_SPREAD',
             long_strike=180, short_strike=205, quantity=1, entry_debit=14.20, current_spread_value=17.95,
             underlying_price=216.48, iv=0.55, opened='2026-05-13', closed='2026-05-29',
             long_action='SELL TO CLOSE', short_action='BUY TO CLOSE', long_exit=41.57, short_exit=23.62,
             notes='May-29-2026 closed execution: STC 180C at 41.57 and BTC 205C at 23.62'),
        # Example open position placeholders
        dict(trade_id='AMD_EXAMPLE_OPEN', symbol='AMD', expiration='2026-08-21', structure='BULL_CALL_SPREAD',
             long_strike=160, short_strike=180, quantity=2, entry_debit=8.50, current_spread_value=10.05,
             underlying_price=174.00, iv=0.50, opened='2026-05-20', closed='',
             long_action='BUY TO OPEN', short_action='SELL TO OPEN', long_exit=np.nan, short_exit=np.nan,
             notes='Open example for engine demonstration'),
    ])

raw_df = sample_bch_book()
display(raw_df)

,trade_id,symbol,expiration,structure,long_strike,short_strike,quantity,entry_debit,current_spread_value,underlying_price,iv,opened,closed,long_action,short_action,long_exit,short_exit,notes
0,NVDA_20260821_180_205_BCS,NVDA,2026-08-21,BULL_CALL_SPREAD,180,205,1,14.2,17.95,216.48,0.55,2026-05-13,2026-05-29,SELL TO CLOSE,BUY TO CLOSE,41.57,23.62,May-29-2026 closed execution: STC 180C at 41.5...
1,AMD_EXAMPLE_OPEN,AMD,2026-08-21,BULL_CALL_SPREAD,160,180,2,8.5,10.05,174.00,0.50,2026-05-20,,BUY TO OPEN,SELL TO OPEN,NaN,NaN,Open example for engine demonstration


## 2. Position Cleaner
The resolver uses execution evidence first. If a matching long leg is sold to close and the short leg is bought to close, the spread is closed regardless of any older “ACTIVE” wording.

In [3]:
# ============================================================
# 2. POSITION CLEANER / STATUS RESOLVER
# ============================================================

CLOSED_SIGNALS = [
    'sold to close', 'sell to close', 'stc',
    'bought to close', 'buy to close', 'btc',
    'closed execution', 'closed the spread correctly',
    'net spread exit value realized', 'realized return'
]

OPEN_SIGNALS = ['buy to open', 'sell to open', 'active', 'open position', 'position status']


def normalize_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip().lower()


def resolve_trade_status(row):
    """Hard override logic: confirmed closing execution > older active language."""
    long_action = normalize_text(row.get('long_action', ''))
    short_action = normalize_text(row.get('short_action', ''))
    notes = normalize_text(row.get('notes', ''))
    closed_date = normalize_text(row.get('closed', ''))
    qty = row.get('quantity', np.nan)

    # Spread close evidence: long call sold to close + short call bought to close.
    long_closed = ('sell to close' in long_action) or ('sold to close' in long_action) or (long_action == 'stc')
    short_closed = ('buy to close' in short_action) or ('bought to close' in short_action) or (short_action == 'btc')

    if long_closed and short_closed:
        return 'CLOSED'
    if closed_date not in ['', 'nan', 'nat', 'none']:
        return 'CLOSED'
    if any(sig in notes for sig in CLOSED_SIGNALS):
        return 'CLOSED'
    if pd.notna(qty) and float(qty) == 0:
        return 'CLOSED'
    return 'OPEN'


def clean_positions(df):
    out = df.copy()
    for c in ['expiration','opened','closed']:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors='coerce')
    numeric_cols = ['long_strike','short_strike','quantity','entry_debit','current_spread_value','underlying_price','iv','long_exit','short_exit']
    for c in numeric_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')
    out['status'] = out.apply(resolve_trade_status, axis=1)
    out['dte'] = (out['expiration'] - TODAY).dt.days.clip(lower=0)
    out['spread_width'] = out['short_strike'] - out['long_strike']
    out['capital_deployed'] = out['entry_debit'] * CONTRACT_MULTIPLIER * out['quantity'].abs()
    return out

positions = clean_positions(raw_df)
display(positions[['trade_id','symbol','expiration','quantity','entry_debit','current_spread_value','status','dte','capital_deployed']])

,trade_id,symbol,expiration,quantity,entry_debit,current_spread_value,status,dte,capital_deployed
0,NVDA_20260821_180_205_BCS,NVDA,2026-08-21,1,14.2,17.95,CLOSED,83,1420.0
1,AMD_EXAMPLE_OPEN,AMD,2026-08-21,2,8.5,10.05,OPEN,83,1700.0


## 3. Greeks Engine
This calculates Black-Scholes Greeks for each leg and nets them at trade and book level. Use live broker/API Greeks when available; use this as the independent control model.

In [4]:
# ============================================================
# 3. GREEKS ENGINE
# ============================================================

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def norm_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

def bs_call_greeks(S, K, T, r=RISK_FREE_RATE, sigma=DEFAULT_IV):
    """Returns call delta, gamma, theta/day, vega per 1% IV move."""
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        intrinsic_delta = 1.0 if S > K else 0.0
        return dict(delta=intrinsic_delta, gamma=0.0, theta=0.0, vega=0.0)
    d1 = (math.log(S/K) + (r + 0.5*sigma*sigma)*T) / (sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    delta = norm_cdf(d1)
    gamma = norm_pdf(d1)/(S*sigma*math.sqrt(T))
    theta_annual = -(S*norm_pdf(d1)*sigma)/(2*math.sqrt(T)) - r*K*math.exp(-r*T)*norm_cdf(d2)
    theta_day = theta_annual / 365.0
    vega_1pct = S * norm_pdf(d1) * math.sqrt(T) / 100.0
    return dict(delta=delta, gamma=gamma, theta=theta_day, vega=vega_1pct)


def attach_trade_greeks(df):
    out = df.copy()
    rows = []
    for _, r in out.iterrows():
        S = float(r['underlying_price'])
        T = max(float(r['dte']) / 365.0, 1/365)
        iv = float(r['iv']) if pd.notna(r['iv']) else DEFAULT_IV
        q = float(r['quantity'])
        lg = bs_call_greeks(S, float(r['long_strike']), T, sigma=iv)
        sg = bs_call_greeks(S, float(r['short_strike']), T, sigma=iv)
        # Bull call spread: long lower strike call, short higher strike call.
        net = {k: (lg[k] - sg[k]) * CONTRACT_MULTIPLIER * q for k in ['delta','gamma','theta','vega']}
        rows.append(net)
    g = pd.DataFrame(rows, index=out.index).add_prefix('net_')
    return pd.concat([out, g], axis=1)

positions_greeks = attach_trade_greeks(positions)
open_positions = positions_greeks[positions_greeks['status'].eq('OPEN')].copy()
closed_positions = positions_greeks[positions_greeks['status'].eq('CLOSED')].copy()

display(positions_greeks[['trade_id','symbol','status','net_delta','net_gamma','net_theta','net_vega']])

book_greeks = open_positions[['net_delta','net_gamma','net_theta','net_vega']].sum().to_frame('BCH_BOOK_TOTAL').T
print('BOOK-LEVEL GREEKS: OPEN POSITIONS ONLY')
display(book_greeks)

,trade_id,symbol,status,net_delta,net_gamma,net_theta,net_vega
0,NVDA_20260821_180_205_BCS,NVDA,CLOSED,16.162937,-0.174545,3.153003,-10.230358
1,AMD_EXAMPLE_OPEN,AMD,OPEN,37.679118,-0.237553,1.903395,-8.177382


BOOK-LEVEL GREEKS: OPEN POSITIONS ONLY


,net_delta,net_gamma,net_theta,net_vega
BCH_BOOK_TOTAL,37.679118,-0.237553,1.903395,-8.177382


## 4. Exit Engine
Ranks every open position by proximity to BCH targets: 20%, 36%, max extraction, and stop-loss.

In [5]:
# ============================================================
# 4. EXIT ENGINE
# ============================================================

TARGET_1_ROI = 0.20
TARGET_2_ROI = 0.36
STOP_ROI = -0.25
MAX_EXTRACTION_CAPTURE = 0.992  # 99.2% of max spread value


def attach_exit_metrics(df):
    out = df.copy()
    out['current_profit_per_spread'] = (out['current_spread_value'] - out['entry_debit']) * CONTRACT_MULTIPLIER
    out['current_profit_total'] = out['current_profit_per_spread'] * out['quantity'].abs()
    out['current_roi'] = (out['current_spread_value'] - out['entry_debit']) / out['entry_debit']
    out['target_20_value'] = out['entry_debit'] * (1 + TARGET_1_ROI)
    out['target_36_value'] = out['entry_debit'] * (1 + TARGET_2_ROI)
    out['stop_value'] = out['entry_debit'] * (1 + STOP_ROI)
    out['max_extract_value'] = out['spread_width'] * MAX_EXTRACTION_CAPTURE
    out['distance_to_20'] = out['target_20_value'] - out['current_spread_value']
    out['distance_to_36'] = out['target_36_value'] - out['current_spread_value']
    out['distance_to_stop'] = out['current_spread_value'] - out['stop_value']
    out['distance_to_max_extract'] = out['max_extract_value'] - out['current_spread_value']

    def command(r):
        if r['status'] == 'CLOSED':
            return 'ARCHIVE / PERFORMANCE DATA'
        if r['current_roi'] <= STOP_ROI:
            return 'EXIT / STOP CONTROL'
        if r['current_roi'] >= TARGET_2_ROI:
            return 'EXIT / GOLDEN EXPANSION'
        if r['current_roi'] >= TARGET_1_ROI:
            return 'TAKE PROFIT / BASE EXTRACTION'
        if r['distance_to_20'] <= 0.50:
            return 'WATCH CLOSELY / NEAR 20%'
        return 'HOLD / MONITOR'

    out['bch_command'] = out.apply(command, axis=1)
    out['urgency_score'] = np.select(
        [out['bch_command'].str.contains('EXIT'), out['bch_command'].str.contains('TAKE PROFIT'), out['bch_command'].str.contains('WATCH')],
        [100, 85, 65], default=35
    )
    return out.sort_values(['status','urgency_score'], ascending=[False, False])

exit_book = attach_exit_metrics(positions_greeks)
cols = ['trade_id','symbol','status','quantity','entry_debit','current_spread_value','current_profit_total','current_roi','bch_command','urgency_score']
display(exit_book[cols].style.format({'current_profit_total':'${:,.2f}','current_roi':'{:.2%}'}))

,trade_id,symbol,status,quantity,entry_debit,current_spread_value,current_profit_total,current_roi,bch_command,urgency_score
1,AMD_EXAMPLE_OPEN,AMD,OPEN,2,8.500000,10.050000,$310.00,18.24%,WATCH CLOSELY / NEAR 20%,65
0,NVDA_20260821_180_205_BCS,NVDA,CLOSED,1,14.200000,17.950000,$375.00,26.41%,ARCHIVE / PERFORMANCE DATA,35


## 5. Heat Limit Engine
This flags concentration risk before a single trade becomes a portfolio-level problem.

In [6]:
# ============================================================
# 5. HEAT LIMIT ENGINE
# ============================================================

HEAT_LIMITS = {
    'max_ticker_capital_pct': 0.40,
    'max_expiration_capital_pct': 0.50,
    'max_single_trade_capital_pct': 0.25,
    'max_book_delta_abs': 250,
    'max_negative_theta_per_day': -75,
}


def heat_limit_report(open_df):
    alerts = []
    if open_df.empty:
        return pd.DataFrame([dict(level='INFO', metric='OPEN BOOK', value=0, limit='n/a', message='No open positions. Book risk is flat.')])

    total_cap = open_df['capital_deployed'].sum()
    by_symbol = open_df.groupby('symbol')['capital_deployed'].sum() / total_cap
    by_exp = open_df.groupby('expiration')['capital_deployed'].sum() / total_cap

    for sym, pct in by_symbol.items():
        if pct > HEAT_LIMITS['max_ticker_capital_pct']:
            alerts.append(dict(level='RED', metric='TICKER CONCENTRATION', value=f'{pct:.1%}', limit=f"{HEAT_LIMITS['max_ticker_capital_pct']:.0%}", message=f'{sym} exceeds ticker capital heat limit.'))
    for exp, pct in by_exp.items():
        if pct > HEAT_LIMITS['max_expiration_capital_pct']:
            alerts.append(dict(level='YELLOW', metric='EXPIRATION CONCENTRATION', value=f'{pct:.1%}', limit=f"{HEAT_LIMITS['max_expiration_capital_pct']:.0%}", message=f'{exp.date()} exceeds expiration heat limit.'))
    for _, r in open_df.iterrows():
        pct = r['capital_deployed'] / total_cap
        if pct > HEAT_LIMITS['max_single_trade_capital_pct']:
            alerts.append(dict(level='YELLOW', metric='SINGLE TRADE SIZE', value=f'{pct:.1%}', limit=f"{HEAT_LIMITS['max_single_trade_capital_pct']:.0%}", message=f"{r['trade_id']} is oversized versus book."))

    total_delta = open_df['net_delta'].sum()
    total_theta = open_df['net_theta'].sum()
    if abs(total_delta) > HEAT_LIMITS['max_book_delta_abs']:
        alerts.append(dict(level='RED', metric='BOOK DELTA', value=f'{total_delta:.1f}', limit=f"±{HEAT_LIMITS['max_book_delta_abs']}", message='Directional exposure is above BCH heat limit.'))
    if total_theta < HEAT_LIMITS['max_negative_theta_per_day']:
        alerts.append(dict(level='RED', metric='BOOK THETA', value=f'{total_theta:.2f}/day', limit=HEAT_LIMITS['max_negative_theta_per_day'], message='Daily time decay is above BCH heat limit.'))

    if not alerts:
        alerts.append(dict(level='GREEN', metric='BOOK HEAT', value='OK', limit='Within limits', message='No heat-limit breaches detected.'))
    return pd.DataFrame(alerts)

heat_report = heat_limit_report(open_positions)
display(heat_report)

,level,metric,value,limit,message
0,RED,TICKER CONCENTRATION,100.0%,40%,AMD exceeds ticker capital heat limit.
1,YELLOW,EXPIRATION CONCENTRATION,100.0%,50%,2026-08-21 exceeds expiration heat limit.
2,YELLOW,SINGLE TRADE SIZE,100.0%,25%,AMD_EXAMPLE_OPEN is oversized versus book.


## 6. BCH Command Dashboard
Final output: what is closed, what is open, what needs action, and where the book is exposed.

In [7]:
# ============================================================
# 6. BCH COMMAND DASHBOARD
# ============================================================

def bch_command_dashboard(exit_df, heat_df):
    open_df = exit_df[exit_df['status'].eq('OPEN')].copy()
    closed_df = exit_df[exit_df['status'].eq('CLOSED')].copy()

    realized_profit = closed_df['current_profit_total'].sum() if not closed_df.empty else 0
    open_profit = open_df['current_profit_total'].sum() if not open_df.empty else 0
    total_open_capital = open_df['capital_deployed'].sum() if not open_df.empty else 0

    display(Markdown(f"""
# BCH COMMAND DASHBOARD
**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Management Verdict
- **Closed trades:** {len(closed_df)}
- **Open trades:** {len(open_df)}
- **Realized/closed trade profit model:** ${realized_profit:,.2f}
- **Open P/L model:** ${open_profit:,.2f}
- **Open capital at risk:** ${total_open_capital:,.2f}

## Rule
Closed trades are **performance data**. Open trades are **risk inventory**.
"""))

    display(Markdown('## Daily Action Queue'))
    display(exit_df[['trade_id','symbol','status','current_roi','current_profit_total','bch_command','urgency_score']]
            .sort_values('urgency_score', ascending=False)
            .style.format({'current_roi':'{:.2%}','current_profit_total':'${:,.2f}'}))

    display(Markdown('## Book-Level Greeks: Open Positions Only'))
    if open_df.empty:
        display(pd.DataFrame([dict(net_delta=0, net_gamma=0, net_theta=0, net_vega=0)]))
    else:
        display(open_df[['net_delta','net_gamma','net_theta','net_vega']].sum().to_frame('BCH_BOOK_TOTAL').T)

    display(Markdown('## Heat Limit Report'))
    display(heat_df)

bch_command_dashboard(exit_book, heat_report)


# BCH COMMAND DASHBOARD
**Generated:** 2026-05-30 16:14:48

## Management Verdict
- **Closed trades:** 1
- **Open trades:** 1
- **Realized/closed trade profit model:** $375.00
- **Open P/L model:** $310.00
- **Open capital at risk:** $1,700.00

## Rule
Closed trades are **performance data**. Open trades are **risk inventory**.


## Daily Action Queue

,trade_id,symbol,status,current_roi,current_profit_total,bch_command,urgency_score
1,AMD_EXAMPLE_OPEN,AMD,OPEN,18.24%,$310.00,WATCH CLOSELY / NEAR 20%,65
0,NVDA_20260821_180_205_BCS,NVDA,CLOSED,26.41%,$375.00,ARCHIVE / PERFORMANCE DATA,35


## Book-Level Greeks: Open Positions Only

,net_delta,net_gamma,net_theta,net_vega
BCH_BOOK_TOTAL,37.679118,-0.237553,1.903395,-8.177382


## Heat Limit Report

,level,metric,value,limit,message
0,RED,TICKER CONCENTRATION,100.0%,40%,AMD exceeds ticker capital heat limit.
1,YELLOW,EXPIRATION CONCENTRATION,100.0%,50%,2026-08-21 exceeds expiration heat limit.
2,YELLOW,SINGLE TRADE SIZE,100.0%,25%,AMD_EXAMPLE_OPEN is oversized versus book.


## 7. Export Reports
Creates CSV files for the cleaned book, action queue, and heat report.

In [8]:
# ============================================================
# 7. EXPORT REPORTS
# ============================================================

def run_bch_engine():
    exit_book.to_csv('BCH_Command_Action_Queue.csv', index=False)
    heat_report.to_csv('BCH_Heat_Limit_Report.csv', index=False)
    positions_greeks.to_csv('BCH_Cleaned_Positions_With_Greeks.csv', index=False)

    print("BCH engine completed.")
    print("Exported:")
    print("- BCH_Command_Action_Queue.csv")
    print("- BCH_Heat_Limit_Report.csv")
    print("- BCH_Cleaned_Positions_With_Greeks.csv")

run_bch_engine()

BCH engine completed.
Exported:
- BCH_Command_Action_Queue.csv
- BCH_Heat_Limit_Report.csv
- BCH_Cleaned_Positions_With_Greeks.csv
